# 08 Detection baseline (M2-1)
Runs on Colab, **fresh runtime**, CPU. Settings come from PLAN.md (M2, decisions of 2026-11-15): `model.val` with imgsz 640, conf 0.001, iou 0.6, seed 0, batch 1, device cpu.

- **Cells 1-6:** load baseline A (Babblu2821/alpr-plate-detector) before Drive is mounted, record its MD5, and time 20 images per set. `model.val` runs there only for timing; its metrics are discarded.
- **Cells 7-8:** the full evaluation of the sets in `RUN`. It prints numbers only (no images, no plate text) and saves one JSON per set under `reports/m2/` on Drive; a finished set is skipped.

In [ ]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition
uv pip install --system -q ultralytics huggingface_hub

In [ ]:
import os
os.environ['YOLO_VERBOSE'] = 'False'   # before the ultralytics import: its val summary line (metrics) and progress bars stay silent
import csv, json, platform, random, shutil, statistics, tarfile, time
from pathlib import Path, PurePosixPath
import torch
import ultralytics
import yaml
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from ultralytics import YOLO
from alpr import ccpd, freeze

In [ ]:
# Parameters
SEED = 0
REPO_A, WEIGHTS_A = 'Babblu2821/alpr-plate-detector', 'best.pt'   # candidate A (PLAN.md)
N_SAMPLE, REPEATS = 20, 3          # images per set, timing passes per image
IMGSZ, CONF, IOU = 640, 0.001, 0.6  # same for every detector
DRIVE = '/content/drive/MyDrive/alpr'
LOCAL, WORK = '/content/tars', '/content/work'
SETS = {   # name -> (tar, MD5 recorded in PLAN.md, split folder, images in the full set)
    'us': ('us_yolo.tar', '1ce491ae6a7accefdb0b9156a30f2acc', 'test', 105),
    'ccpd': ('ccpd_yolo.tar', 'bd44986acd6691a4f90fbea1e6017682', 'heldout', 300),
    'uc3m_395': ('uc3m_yolo.tar', 'e3ef306731f33095bac983338c6adb04', 'test', 395),
}
UC3M_ALL = 1975
RUN = ['us', 'ccpd', 'uc3m_395', 'uc3m_all']   # sets evaluated by the full-run cell, small first; list one to run one set
REPORTS = f'{DRIVE}/reports/m2'
EVAL = {   # set -> tar, evaluated splits, expected images, frozen manifest (PLAN.md, M1b-7)
    'us': dict(tar='us_yolo.tar', splits=('test',), n=105, manifest='us_test'),
    'ccpd': dict(tar='ccpd_yolo.tar', splits=('heldout',), n=300, manifest='ccpd_heldout'),
    'uc3m_395': dict(tar='uc3m_yolo.tar', splits=('test',), n=395, manifest='uc3m_all'),   # the authors' test split: a subset of the manifest
    'uc3m_all': dict(tar='uc3m_yolo.tar', splits=('train', 'test'), n=1975, manifest='uc3m_all'),
}
MANIFEST_MD5 = {'us_test': '8b83bd4b5074dd7714908dc9a778f8ec', 'ccpd_heldout': 'a12032f8f9d2e8c16bf4cc95c05a269b', 'uc3m_all': 'decf0504f9415596e500969c4ac616d2'}

In [ ]:
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
print('CPU:', cpu, '| cores:', os.cpu_count(), '| python', platform.python_version(), '| ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('GPU visible:', torch.cuda.is_available(), '(all runs below use device=cpu)')

In [ ]:
# 1. Weights BEFORE Drive is mounted: download to the VM disk, record the MD5, load the model.
assert not os.path.ismount('/content/drive'), 'restart the runtime: Drive must not be mounted yet'
info = HfApi().model_info(REPO_A)
print('HF commit:', info.sha, '| licence tag:', info.card_data.license if info.card_data else None)
wpath = hf_hub_download(REPO_A, WEIGHTS_A, revision=info.sha, local_dir=f'{WORK}/weights')
W_MD5 = ccpd.file_md5(wpath)
print(f'{WEIGHTS_A}: {Path(wpath).stat().st_size / 1e6:.2f} MB, MD5 {W_MD5}')
model = YOLO(wpath)
assert len(model.names) == 1, model.names   # single class: val matches class 0 of our labels
print('class names:', model.names, '| task:', model.task)

In [ ]:
# 2. Only now mount Drive and copy the weights there (MD5 checked).
drive.mount('/content/drive')
dst = Path(DRIVE, 'weights', 'baseline_A_best.pt')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(wpath, dst)
assert ccpd.file_md5(dst) == W_MD5, 'Drive copy differs'
print('copied to', dst, '| record in PLAN.md: HF commit', info.sha, 'MD5', W_MD5)

In [ ]:
# 3. Test tars to the VM disk, MD5 against PLAN.md (time reported: part of the full-run cost).
Path(LOCAL).mkdir(exist_ok=True)
for name, (fn, md5, _, _) in SETS.items():
    src, d = Path(DRIVE, 'archives', fn), Path(LOCAL, fn)
    assert src.exists(), f'{fn} not found'
    t0 = time.perf_counter()
    if not d.exists():
        shutil.copyfile(src, d)
    t1 = time.perf_counter()
    got = ccpd.file_md5(d)
    assert got == md5, f'{name}: MD5 {got} differs from PLAN.md'
    print(f'{name}: copy {t1 - t0:.0f} s, MD5 check {time.perf_counter() - t1:.0f} s, ok')

In [ ]:
# 4. Extract a seeded sample of N_SAMPLE images + labels per set, and write a data yaml for each
# (train and val both point at the sample: Ultralytics requires both keys, only val is used and the metrics are discarded).
SAMPLE = {}
for name, (fn, _, split, _) in SETS.items():
    root = Path(WORK, 'sample', name)
    (root / 'images').mkdir(parents=True, exist_ok=True)
    (root / 'labels').mkdir(exist_ok=True)
    with tarfile.open(Path(LOCAL, fn)) as t:
        names = sorted(m.name for m in t.getmembers() if m.isfile() and m.name.startswith(f'images/{split}/'))
        for n in random.Random(SEED).sample(names, N_SAMPLE):
            stem = Path(n).stem
            lab = f'labels/{split}/{stem}.txt'
            (root / 'images' / Path(n).name).write_bytes(t.extractfile(n).read())
            (root / 'labels' / f'{stem}.txt').write_bytes(t.extractfile(lab).read())
    (root / 'data.yaml').write_text(f'path: {root}\ntrain: images\nval: images\nnames:\n  0: plate\n')
    SAMPLE[name] = sorted((root / 'images').iterdir())
    print(name, len(SAMPLE[name]), 'images')

In [ ]:
# 5. Timing: decode and predict per image (warm-up excluded), then wall time of model.val on the sample (metrics discarded, not printed).
def spread(v):
    q = statistics.quantiles(v, n=4)
    return f'median {statistics.median(v) * 1000:.0f} ms | min {min(v) * 1000:.0f} | IQR {q[0] * 1000:.0f}-{q[2] * 1000:.0f} | max {max(v) * 1000:.0f}'

full = {'us': 105, 'ccpd': 300, 'uc3m_395': 395}
model.predict(str(SAMPLE['us'][0]), imgsz=IMGSZ, device='cpu', verbose=False)   # warm-up
for name, files in SAMPLE.items():
    dec, pred = [], []
    for f in files:
        t0 = time.perf_counter()
        with Image.open(f) as im:
            im.load()
        dec.append(time.perf_counter() - t0)
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            model.predict(str(f), imgsz=IMGSZ, conf=CONF, iou=IOU, device='cpu', verbose=False)
            pred.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    r = model.val(data=str(SAMPLE[name][0].parents[1] / 'data.yaml'), imgsz=IMGSZ, conf=CONF, iou=IOU, device='cpu', batch=1,
                  workers=2, plots=False, verbose=False, seed=SEED, project=f'{WORK}/runs', name='sample', exist_ok=True)
    val_s = time.perf_counter() - t0
    n = full[name]
    print(f'{name}: decode {spread(dec)}\n    predict {spread(pred)}\n    val of {N_SAMPLE}: {val_s:.0f} s -> full {n} images ~ {val_s / N_SAMPLE * n / 60:.1f} min at imgsz {IMGSZ}')
print(f'uc3m all {UC3M_ALL} images: scale the uc3m_395 line by {UC3M_ALL / 395:.1f}')

In [ ]:
# 6. Training image size recorded in the checkpoint (raw train_args: model.args merges in defaults, so it cannot tell "recorded" from "default").
ta = getattr(model, 'ckpt', None)
rec = ta.get('train_args', {}).get('imgsz') if isinstance(ta, dict) else None
print('recorded training imgsz:', rec if rec is not None else 'not recorded (no second size is run, nothing is guessed)')
SIZES = [IMGSZ]
if isinstance(rec, int) and rec != IMGSZ:
    SIZES.append(rec)   # a second, labelled run at the recorded size
elif rec is not None and rec != IMGSZ:
    print('recorded in a form not used here:', type(rec).__name__, '- no second size is run')

In [ ]:
# 7. Helpers for the full evaluation. evaluate() is the one function every detector goes through (settings from PLAN.md).
VERIFIED = set()
def ensure_tar(fn):
    # copy to the VM disk and check the MD5 recorded in PLAN.md (once per runtime)
    md5 = next(m for f, m, _, _ in SETS.values() if f == fn)
    d = Path(LOCAL, fn)
    if fn not in VERIFIED:
        t0 = time.perf_counter()
        Path(LOCAL).mkdir(exist_ok=True)
        if not d.exists():
            shutil.copyfile(Path(DRIVE, 'archives', fn), d)
        assert ccpd.file_md5(d) == md5, f'{fn}: MD5 differs from PLAN.md'
        VERIFIED.add(fn)
        print(f'{fn}: ready in {time.perf_counter() - t0:.0f} s (copy if needed + MD5 check)')
    return d

def check_frozen(name):
    # the evaluated image names and bytes must match the frozen manifest on Drive (whose own MD5 is in PLAN.md)
    e = EVAL[name]
    mpath = Path(DRIVE, 'frozen', f"{e['manifest']}_manifest.csv")
    assert ccpd.file_md5(mpath) == MANIFEST_MD5[e['manifest']], f'{mpath.name}: MD5 differs from PLAN.md'
    with_split = e['manifest'] == 'uc3m_all'   # the UC3M manifest names are 'split/file'
    if name == 'uc3m_395':   # a subset of the manifest: names and MD5s must all be in it
        got = freeze.tar_md5s(Path(LOCAL, e['tar']), e['splits'], with_split)
        with open(mpath, newline='', encoding='utf-8') as f:
            man = {r['name']: r['md5'] for r in csv.DictReader(f)}
        assert len(got) == e['n'] and all(man.get(k) == v for k, v in got.items()), f'{name}: images differ from the frozen manifest'
        return len(got)
    n = freeze.verify(Path(LOCAL, e['tar']), e['splits'], with_split, mpath)   # exact match of the whole manifest
    assert n == e['n'], f'{name}: {n} images, expected {e["n"]}'
    return n

def extract_eval(name):
    # extract only the evaluated splits (images/<split>, labels/<split> as in the tar) and write the data yaml: train and val both point at the evaluated folder(s)
    e = EVAL[name]
    root = Path(WORK, 'eval', Path(e['tar']).stem)
    with tarfile.open(Path(LOCAL, e['tar'])) as t:
        for m in t:
            p = PurePosixPath(m.name)
            if m.isfile() and len(p.parts) == 3 and p.parts[0] in ('images', 'labels') and p.parts[1] in e['splits']:
                dst = root / m.name
                if not dst.exists() or dst.stat().st_size != m.size:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    dst.write_bytes(t.extractfile(m).read())
    for s in e['splits']:
        ni, nl = len(list((root / 'images' / s).iterdir())), len(list((root / 'labels' / s).iterdir()))
        assert ni == nl, f'{name}/{s}: {ni} images, {nl} label files'
    assert sum(len(list((root / 'images' / s).iterdir())) for s in e['splits']) == e['n'], f'{name}: image count differs from {e["n"]}'
    dirs = [f'images/{s}' for s in e['splits']]
    val = dirs[0] if len(dirs) == 1 else '[' + ', '.join(dirs) + ']'
    y = root / f'data_{name}.yaml'
    y.write_text(f'path: {root}\ntrain: {val}\nval: {val}\nnames:\n  0: plate\n')
    return y

def evaluate(model, data_yaml, set_name, imgsz=IMGSZ):
    # model.val with the settings declared in PLAN.md; returns numbers only
    cfg = yaml.safe_load(Path(data_yaml).read_text())
    vals = cfg['val'] if isinstance(cfg['val'], list) else [cfg['val']]
    n_img = sum(1 for v in vals for f in Path(cfg['path'], v).iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    t0 = time.perf_counter()
    r = model.val(data=str(data_yaml), imgsz=imgsz, conf=CONF, iou=IOU, device='cpu', batch=1, workers=2, plots=False, verbose=False,
                  seed=SEED, project=f'{WORK}/runs', name=f'full_{set_name}', exist_ok=True)
    wall = time.perf_counter() - t0
    return {'set': set_name, 'imgsz': imgsz, 'images': n_img, 'instances': int(r.nt_per_class.sum()), 'mAP50': float(r.box.map50),
            'mAP50-95': float(r.box.map), 'precision': float(r.box.mp), 'recall': float(r.box.mr), 'val_seconds': round(wall, 1)}

In [ ]:
# 8. Full evaluation of baseline A on the sets in RUN (numbers only). A set whose JSON already exists on Drive is skipped, so a disconnect loses nothing finished.
Path(REPORTS).mkdir(parents=True, exist_ok=True)
for size in SIZES:
    for name in RUN:
        label = f'{name} at imgsz {size}' + (' (recorded training size)' if size != IMGSZ else '')
        out = Path(REPORTS, f'{name}_imgsz{size}.json')
        if out.exists():
            res, note = json.loads(out.read_text()), ' [skipped: JSON exists]'
        else:
            ensure_tar(EVAL[name]['tar'])
            check_frozen(name)
            res = evaluate(model, extract_eval(name), name, size)
            res.update(weights_md5=W_MD5, hf_commit=info.sha, ultralytics=ultralytics.__version__, conf=CONF, iou=IOU, seed=SEED, device='cpu', batch=1, cpu=cpu)
            part = out.with_name(out.name + '.part')
            part.write_text(json.dumps(res, indent=1))
            part.replace(out)
            note = ''
        print(f"{label}: images {res['images']} | instances {res['instances']} | mAP50 {res['mAP50']:.4f} | mAP50-95 {res['mAP50-95']:.4f} | "
              f"P {res['precision']:.4f} | R {res['recall']:.4f} | val {res['val_seconds']:.0f} s{note}")